# Step 4 — DistilBERT Sentiment Model

Project 2 (HPDP, SECP3133) — Real-Time Sentiment Analysis on Grab Google Play Reviews

**Owner:** Syahmi (NLP & Model Engineer) · **Model category:** Transformer (§6.3)

Fine-tunes **`distilbert-base-multilingual-cased`** — multilingual because Grab reviews mix
English and Malay ("sangat mengecewa", "tak boleh guna", etc.).

Why DistilBERT:
- Pre-trained contextual understanding — knows "sick" is negative in "app is sick of crashing"
  but positive in "this app is sick!" depending on surrounding words
- 40% smaller / 60% faster than full BERT, fits comfortably on a Colab T4

Imbalance handled with a **class-weighted cross-entropy loss** (custom Trainer).
Same train/val/test split from Step 1.

> **GPU is mandatory.** `Runtime -> Change runtime type -> T4 GPU`. On CPU this takes hours.

## 0. Install + setup

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score)

MODEL_NAME = 'distilbert-base-multilingual-cased'
MAXLEN = 128
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
LABELS = ['Negative', 'Neutral', 'Positive']

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: no GPU detected. Training will be very slow. Enable T4 GPU in Colab.')

def find_file(stem):
    """Locate train/val/test, csv or xlsx. Includes Colab root (/content)."""
    roots = ['.', 'data', '/content', '/content/data', '../data']
    for r in roots:
        for ext in ('csv', 'xlsx'):
            p = f'{r}/{stem}.{ext}'
            if os.path.exists(p):
                return p
    raise FileNotFoundError(
        f'{stem}.(csv|xlsx) not found. On Colab, upload {stem}.csv to the file '
        'panel (it lands in /content/), then re-run this cell.')

def load(stem):
    p = find_file(stem)
    df = pd.read_excel(p) if p.endswith('.xlsx') else pd.read_csv(p)
    df['final_clean_text'] = df['final_clean_text'].fillna('')
    print(f'  {stem}: {len(df):,} rows  <- {p}')
    return df

MODELS_DIR = 'models'; os.makedirs(MODELS_DIR, exist_ok=True)
print('Loading splits...')
train = load('train'); val = load('val'); test = load('test')

## 1. Tokenizer + Dataset

DistilBERT brings its own subword tokenizer — no manual vocabulary needed. We wrap the splits in
a `torch.utils.data.Dataset` that tokenizes on the fly.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class GrabDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), truncation=True, padding='max_length',
                             max_length=MAXLEN)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(int(self.labels[i]))
        return item

train_ds = GrabDataset(train['final_clean_text'], train['sentiment_label'])
val_ds   = GrabDataset(val['final_clean_text'],   val['sentiment_label'])
test_ds  = GrabDataset(test['final_clean_text'],  test['sentiment_label'])
print('Datasets built.')

## 2. Class-weighted Trainer

Subclass `Trainer` to inject class weights into the cross-entropy loss, so the rare Neutral
class is not drowned out. Weights are computed `'balanced'` from the training labels.

In [ ]:
cw = compute_class_weight('balanced', classes=np.array([0, 1, 2]),
                          y=train['sentiment_label'].astype(int))
class_weights = torch.tensor(cw, dtype=torch.float).to(device)
print('Class weights:', {LABELS[i]: round(float(w), 2) for i, w in enumerate(cw)})

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=class_weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

## 3. Fine-tune

3 epochs at lr 2e-5 is the standard recipe for DistilBERT fine-tuning. `fp16=True` uses the T4's
mixed-precision cores for a ~2x speedup. Best checkpoint (by macro-F1) is restored at the end.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {'accuracy': accuracy_score(labels, preds),
            'macro_f1': f1_score(labels, preds, average='macro')}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

args = TrainingArguments(
    output_dir='./distilbert_output',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    fp16=(device == 'cuda'),
    logging_steps=100,
    report_to='none',
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

t0 = time.time()
trainer.train()
train_time = time.time() - t0
print(f'\nFine-tuned in {train_time/60:.1f} min')

## 4. Evaluate on the test set

In [ ]:
t0 = time.time()
pred_out = trainer.predict(test_ds)
infer_ms = (time.time() - t0) / len(test_ds) * 1000
y_pred = pred_out.predictions.argmax(axis=1)
y_test = np.array(test['sentiment_label'].astype(int))

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
print(f'Accuracy : {acc:.4f}')
print(f'Macro-F1 : {macro_f1:.4f}')
print(f'Inference: {infer_ms:.4f} ms/sample\n')
print(classification_report(y_test, y_pred, target_names=LABELS, digits=4))

## 5. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'DistilBERT — Confusion Matrix\nAccuracy={acc:.3f}  Macro-F1={macro_f1:.3f}')
plt.tight_layout(); plt.savefig('distilbert_confusion_matrix.png', dpi=120); plt.show()

## 6. Save model + tokenizer + metrics

`save_pretrained` writes a **directory** (config + weights + tokenizer files), not a single file.

In [ ]:
save_dir = f'{MODELS_DIR}/distilbert'
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

report = classification_report(y_test, y_pred, target_names=LABELS, output_dict=True)
metrics = {
    'model': 'DistilBERT',
    'accuracy': acc, 'macro_f1': macro_f1,
    'macro_precision': report['macro avg']['precision'],
    'macro_recall': report['macro avg']['recall'],
    'f1_negative': report['Negative']['f1-score'],
    'f1_neutral': report['Neutral']['f1-score'],
    'f1_positive': report['Positive']['f1-score'],
    'train_time_s': train_time,
    'inference_ms_per_sample': infer_ms,
}
with open(f'{MODELS_DIR}/metrics_distilbert.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Saved DistilBERT to {save_dir}/ + metrics_distilbert.json')
print('\n[OK] Step 4 complete. All 3 models trained \u2014 ready for Step 5 comparison.')

## 7. (Colab) Download outputs to your computer

DistilBERT saves a **folder** of files (~500 MB), so we zip it first then download
a single archive. The metrics JSON and confusion matrix PNG download separately.

After the zip lands in your browser's Downloads folder, **unzip it into your local
`models/` directory** so you end up with `models/distilbert/` containing
`config.json`, `model.safetensors`, `tokenizer.json`, etc.

In [ ]:
import shutil
from google.colab import files

# Zip the distilbert/ directory into one file for easy download
zip_path = shutil.make_archive(f'{MODELS_DIR}/distilbert', 'zip', save_dir)
print(f'Zipped {save_dir}/ -> {zip_path}')

files.download(zip_path)
files.download(f'{MODELS_DIR}/metrics_distilbert.json')
files.download('distilbert_confusion_matrix.png')